In [1]:
import pandas as pd

In [2]:
import pandas as pd
import random

# Possible values
event_types = ["wedding", "birthday", "corporate"]
cities = ["Bangalore", "Mumbai", "Delhi", "Hyderabad", "Chennai"]

data = []

for _ in range(1200):  # generate 1200 rows
    event = random.choice(event_types)
    city = random.choice(cities)

    # Guests range based on event type
    if event == "wedding":
        guests = random.randint(100, 500)
        base_budget = random.randint(500000, 3000000)
    elif event == "corporate":
        guests = random.randint(50, 300)
        base_budget = random.randint(200000, 1500000)
    else:  # birthday
        guests = random.randint(20, 100)
        base_budget = random.randint(50000, 300000)

    # Cost distribution (realistic % split)
    venue_cost = int(base_budget * random.uniform(0.25, 0.40))
    catering_cost = int(base_budget * random.uniform(0.20, 0.35))
    decoration_cost = int(base_budget * random.uniform(0.10, 0.20))
    entertainment_cost = int(base_budget * random.uniform(0.05, 0.15))

    data.append([
        event, city, guests, base_budget,
        venue_cost, catering_cost,
        decoration_cost, entertainment_cost
    ])

# Create DataFrame
df = pd.DataFrame(data, columns=[
    "event_type", "city", "guests", "budget",
    "venue_cost", "catering_cost", "decoration_cost", "entertainment_cost"
])

# Save to CSV
df.to_csv("event_dataset_1000.csv", index=False)

print("✅ Dataset with 1000+ rows created!")
df.head()

✅ Dataset with 1000+ rows created!


,event_type,city,guests,budget,venue_cost,catering_cost,decoration_cost,entertainment_cost
0,birthday,Delhi,83,208343,75185,60044,30720,23741
1,corporate,Delhi,169,287852,95515,91856,41613,36155
2,wedding,Delhi,135,1228626,325843,275958,210674,81076
3,birthday,Mumbai,52,171363,57440,59527,23475,12270
4,birthday,Delhi,34,271185,90052,82391,31571,22329


In [3]:
df.shape
df.describe()
df["event_type"].value_counts()

,count
event_type,
birthday,416
corporate,397
wedding,387


In [4]:
EVENT_TYPES = ["wedding", "corporate", "birthday"]

VENDOR_CATEGORIES = {
    "wedding": ["venue", "catering", "decoration", "photography", "entertainment"],
    "corporate": ["venue", "catering", "audio_visual", "branding"],
    "birthday": ["venue", "cake", "decoration", "entertainment"]
}

print(EVENT_TYPES)
print(VENDOR_CATEGORIES)

['wedding', 'corporate', 'birthday']
{'wedding': ['venue', 'catering', 'decoration', 'photography', 'entertainment'], 'corporate': ['venue', 'catering', 'audio_visual', 'branding'], 'birthday': ['venue', 'cake', 'decoration', 'entertainment']}


In [5]:
print("🔍 DATA INFO")
df.info()

print("\n📊 SUMMARY")
df.describe()

print("\n🎯 EVENT DISTRIBUTION")
print(df["event_type"].value_counts())

print("\n💰 AVG BUDGET BY EVENT")
print(df.groupby("event_type")["budget"].mean())

🔍 DATA INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   event_type          1200 non-null   object
 1   city                1200 non-null   object
 2   guests              1200 non-null   int64 
 3   budget              1200 non-null   int64 
 4   venue_cost          1200 non-null   int64 
 5   catering_cost       1200 non-null   int64 
 6   decoration_cost     1200 non-null   int64 
 7   entertainment_cost  1200 non-null   int64 
dtypes: int64(6), object(2)
memory usage: 75.1+ KB

📊 SUMMARY

🎯 EVENT DISTRIBUTION
event_type
birthday     416
corporate    397
wedding      387
Name: count, dtype: int64

💰 AVG BUDGET BY EVENT
event_type
birthday     1.827225e+05
corporate    8.870143e+05
wedding      1.727884e+06
Name: budget, dtype: float64


In [6]:
!pip install scikit-learn

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error

In [7]:
le_event = LabelEncoder()
le_city = LabelEncoder()

df['event_type_encoded'] = le_event.fit_transform(df['event_type'])
df['city_encoded'] = le_city.fit_transform(df['city'])

df.head()

,event_type,city,guests,budget,venue_cost,catering_cost,decoration_cost,entertainment_cost,event_type_encoded,city_encoded
0,birthday,Delhi,83,208343,75185,60044,30720,23741,0,2
1,corporate,Delhi,169,287852,95515,91856,41613,36155,1,2
2,wedding,Delhi,135,1228626,325843,275958,210674,81076,2,2
3,birthday,Mumbai,52,171363,57440,59527,23475,12270,0,4
4,birthday,Delhi,34,271185,90052,82391,31571,22329,0,2


In [8]:
X = df[['event_type_encoded', 'city_encoded', 'guests', 'budget']]
y = df['venue_cost']

In [9]:
targets = ['venue_cost', 'catering_cost', 'decoration_cost', 'entertainment_cost']

models = {}

for target in targets:
    y = df[target]

    model = LinearRegression()
    model.fit(X, y)

    models[target] = model

print("✅ All models trained")

✅ All models trained


In [10]:
models = {}

for target in targets:
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = LinearRegression()
    model.fit(X_train, y_train)

    models[target] = model

In [11]:
from sklearn.metrics import mean_absolute_error

for target in targets:
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    error = mean_absolute_error(y_test, preds)

    print(f"{target} MAE: {error}")

venue_cost MAE: 36262.09637407507
catering_cost MAE: 35976.29841397713
decoration_cost MAE: 24313.089583331795
entertainment_cost MAE: 26708.39528846276


In [12]:
event = "wedding"
city = "Bangalore"

event_encoded = le_event.transform([event])[0]
city_encoded = le_city.transform([city])[0]

# Proper DataFrame input
input_data = pd.DataFrame({
    'event_type_encoded': [event_encoded],
    'city_encoded': [city_encoded],
    'guests': [250],
    'budget': [1200000]
})

# Predict all costs
for cost_type, model in models.items():
    prediction = model.predict(input_data)[0]
    print(f"{cost_type}: ₹{int(prediction)}")

venue_cost: ₹396249
catering_cost: ₹335038
decoration_cost: ₹178251
entertainment_cost: ₹120229


In [13]:
!pip install scikit-learn pandas

In [14]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [15]:
data = {
    "text": [
        "I want a wedding in Bangalore",
        "Need birthday decorations",
        "Corporate seminar for employees",
        "Grand wedding with 300 guests",
        "Birthday party for kids",
        "Business conference event",
        "Luxury wedding ceremony",
        "Office annual meeting",
        "Small birthday celebration",
        "Corporate networking event",
        "Wedding reception planning",
        "Birthday cake and balloons",
        "Corporate product launch",
        "Destination wedding planning",
        "Birthday party at home"
    ],

    "event_type": [
        "wedding",
        "birthday",
        "corporate",
        "wedding",
        "birthday",
        "corporate",
        "wedding",
        "corporate",
        "birthday",
        "corporate",
        "wedding",
        "birthday",
        "corporate",
        "wedding",
        "birthday"
    ]
}

df_nlp = pd.DataFrame(data)

df_nlp.head()

,text,event_type
0,I want a wedding in Bangalore,wedding
1,Need birthday decorations,birthday
2,Corporate seminar for employees,corporate
3,Grand wedding with 300 guests,wedding
4,Birthday party for kids,birthday


In [16]:
X = df_nlp["text"]
y = df_nlp["event_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [17]:
model = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', MultinomialNB())
])

model.fit(X_train, y_train)

print("✅ NLP model trained successfully")

✅ NLP model trained successfully


In [18]:
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("🎯 Accuracy:", accuracy)

print("\n📊 Classification Report:\n")
print(classification_report(y_test, predictions))

🎯 Accuracy: 1.0

📊 Classification Report:

              precision    recall  f1-score   support

    birthday       1.00      1.00      1.00         1
   corporate       1.00      1.00      1.00         1
     wedding       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



In [19]:
import joblib

joblib.dump(model, "event_classifier.pkl")

print("✅ Model saved")

✅ Model saved


In [20]:
loaded_model = joblib.load("event_classifier.pkl")

test = "Corporate business meeting"

result = loaded_model.predict([test])

print(result[0])

corporate


In [21]:
user_input = "Need a luxury wedding with music and decorations"

prediction = model.predict([user_input])

print("🎉 Predicted Event Type:", prediction[0])

🎉 Predicted Event Type: wedding


In [22]:
examples = [
    "Birthday party for my daughter",
    "Corporate office meeting",
    "Big fat Indian wedding",
    "Team conference seminar",
    "Cake and balloons for birthday"
]

for text in examples:
    pred = model.predict([text])[0]
    print(f"Text: {text}")
    print(f"Prediction: {pred}")
    print("-" * 40)

Text: Birthday party for my daughter
Prediction: birthday
----------------------------------------
Text: Corporate office meeting
Prediction: corporate
----------------------------------------
Text: Big fat Indian wedding
Prediction: wedding
----------------------------------------
Text: Team conference seminar
Prediction: corporate
----------------------------------------
Text: Cake and balloons for birthday
Prediction: birthday
----------------------------------------


In [23]:
!pip install sentence-transformers faiss-cpu pandas

In [24]:
import pandas as pd
import faiss
import numpy as np

from sentence_transformers import SentenceTransformer

In [25]:
import pandas as pd
import faiss
import numpy as np

from sentence_transformers import SentenceTransformer

In [26]:
knowledge_data = {
    "topic": [
        "Wedding Planning",
        "Corporate Event",
        "Birthday Party",
        "Wedding Budget",
        "Corporate Seminar",
        "Birthday Decorations",
        "Wedding Timeline",
        "Corporate Networking",
        "Kids Birthday",
        "Destination Wedding"
    ],

    "content": [
        "Start wedding planning at least 6 months before the event.",
        "Corporate events require proper scheduling and audio visual setup.",
        "Birthday parties should include cake, games, and decorations.",
        "Allocate 40 percent of wedding budget to venue and catering.",
        "Corporate seminars require registration desks and presentations.",
        "Use balloons, banners, and lighting for birthday decorations.",
        "Finalize wedding venue before sending invitations.",
        "Networking events should include interactive sessions.",
        "Kids birthday events need entertainment and safety measures.",
        "Destination weddings require travel and hotel arrangements."
    ]
}

knowledge_df = pd.DataFrame(knowledge_data)

knowledge_df

,topic,content
0,Wedding Planning,Start wedding planning at least 6 months befor...
1,Corporate Event,Corporate events require proper scheduling and...
2,Birthday Party,"Birthday parties should include cake, games, a..."
3,Wedding Budget,Allocate 40 percent of wedding budget to venue...
4,Corporate Seminar,Corporate seminars require registration desks ...
5,Birthday Decorations,"Use balloons, banners, and lighting for birthd..."
6,Wedding Timeline,Finalize wedding venue before sending invitati...
7,Corporate Networking,Networking events should include interactive s...
8,Kids Birthday,Kids birthday events need entertainment and sa...
9,Destination Wedding,Destination weddings require travel and hotel ...


In [27]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("✅ Embedding model loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded


In [28]:
documents = knowledge_df["content"].tolist()

embeddings = embedding_model.encode(documents)

print("✅ Embeddings created")
print("Embedding shape:", embeddings.shape)

✅ Embeddings created
Embedding shape: (10, 384)


In [29]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("✅ FAISS index created")
print("Total documents:", index.ntotal)

✅ FAISS index created
Total documents: 10


In [30]:
def search_knowledge(query, top_k=3):

    # Convert query to embedding
    query_embedding = embedding_model.encode([query])

    # Search FAISS
    distances, indices = index.search(
        np.array(query_embedding),
        top_k
    )

    results = []

    for idx in indices[0]:
        results.append({
            "topic": knowledge_df.iloc[idx]["topic"],
            "content": knowledge_df.iloc[idx]["content"]
        })

    return results

In [31]:
query = "How should I plan a wedding?"

results = search_knowledge(query)

for i, result in enumerate(results, 1):
    print(f"\n📌 Result {i}")
    print("Topic:", result["topic"])
    print("Content:", result["content"])


📌 Result 1
Topic: Wedding Planning
Content: Start wedding planning at least 6 months before the event.

📌 Result 2
Topic: Wedding Timeline
Content: Finalize wedding venue before sending invitations.

📌 Result 3
Topic: Destination Wedding
Content: Destination weddings require travel and hotel arrangements.


In [32]:
queries = [
    "How to organize business conference?",
    "Ideas for birthday decorations",
    "How to manage wedding budget?"
]

for query in queries:
    print("\n" + "="*50)
    print("🔍 Query:", query)

    results = search_knowledge(query, top_k=2)

    for result in results:
        print("➡", result["content"])


🔍 Query: How to organize business conference?
➡ Corporate seminars require registration desks and presentations.
➡ Corporate events require proper scheduling and audio visual setup.

🔍 Query: Ideas for birthday decorations
➡ Use balloons, banners, and lighting for birthday decorations.
➡ Birthday parties should include cake, games, and decorations.

🔍 Query: How to manage wedding budget?
➡ Allocate 40 percent of wedding budget to venue and catering.
➡ Start wedding planning at least 6 months before the event.


In [33]:
faiss.write_index(index, "event_knowledge.index")

print("✅ FAISS index saved")

✅ FAISS index saved


In [34]:
loaded_index = faiss.read_index("event_knowledge.index")

print("✅ FAISS index loaded")

✅ FAISS index loaded


In [41]:
!pip install transformers torch faiss-cpu pandas scikit-learn

In [42]:
import pandas as pd
import numpy as np
import faiss
import torch

from transformers import AutoTokenizer, AutoModel

In [43]:
knowledge_data = {
    "topic": [
        "Wedding Planning",
        "Corporate Event",
        "Birthday Party",
        "Wedding Budget",
        "Corporate Seminar",
        "Birthday Decorations",
        "Wedding Timeline",
        "Destination Wedding"
    ],

    "content": [
        "Start wedding planning at least 6 months before the event.",
        "Corporate events require proper scheduling and AV setup.",
        "Birthday parties should include cake and decorations.",
        "Allocate 40 percent of wedding budget to venue and catering.",
        "Corporate seminars require registration desks.",
        "Use balloons and lighting for birthday decoration.",
        "Finalize wedding venue before invitations.",
        "Destination weddings require hotel and travel planning."
    ]
}

knowledge_df = pd.DataFrame(knowledge_data)

knowledge_df.head()

,topic,content
0,Wedding Planning,Start wedding planning at least 6 months befor...
1,Corporate Event,Corporate events require proper scheduling and...
2,Birthday Party,Birthday parties should include cake and decor...
3,Wedding Budget,Allocate 40 percent of wedding budget to venue...
4,Corporate Seminar,Corporate seminars require registration desks.


In [44]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
embedding_model = AutoModel.from_pretrained(model_name)

print("✅ Hugging Face model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Hugging Face model loaded


In [45]:
def get_embedding(text):

    inputs = tokenizer(
        text,
        return_tensors='pt',
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = embedding_model(**inputs)

    embeddings = outputs.last_hidden_state.mean(dim=1)

    return embeddings[0].numpy()

In [46]:
documents = knowledge_df["content"].tolist()

embeddings = []

for doc in documents:
    emb = get_embedding(doc)
    embeddings.append(emb)

embeddings = np.array(embeddings)

print("✅ Embeddings created")

✅ Embeddings created


In [47]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("✅ FAISS index created")

✅ FAISS index created


In [48]:
def knowledge_agent(query, top_k=2):

    query_embedding = get_embedding(query)

    query_embedding = np.array([query_embedding])

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for idx in indices[0]:
        results.append(
            knowledge_df.iloc[idx]["content"]
        )

    return results

In [49]:
def budget_agent(total_budget):

    allocation = {
        "venue": int(total_budget * 0.35),
        "catering": int(total_budget * 0.30),
        "decoration": int(total_budget * 0.15),
        "entertainment": int(total_budget * 0.10),
        "miscellaneous": int(total_budget * 0.10)
    }

    return allocation

In [50]:
def planner_agent(event_type):

    plans = {
        "wedding": [
            "Book venue",
            "Hire catering service",
            "Arrange decorations",
            "Send invitations",
            "Book photographer"
        ],

        "birthday": [
            "Order birthday cake",
            "Buy decorations",
            "Arrange games",
            "Prepare guest list"
        ],

        "corporate": [
            "Book conference hall",
            "Arrange projector",
            "Invite speakers",
            "Prepare presentation materials"
        ]
    }

    return plans.get(event_type.lower(), ["No plan available"])

In [51]:
def event_planner_system(event_type, city, budget):

    print("=" * 60)
    print("🎉 FESTIVA PLANNER AI")
    print("=" * 60)

    # Planner Agent
    print("\n📋 EVENT PLAN")

    tasks = planner_agent(event_type)

    for i, task in enumerate(tasks, 1):
        print(f"{i}. {task}")

    # Budget Agent
    print("\n💰 BUDGET BREAKDOWN")

    budget_data = budget_agent(budget)

    for key, value in budget_data.items():
        print(f"{key}: ₹{value}")

    # Knowledge Agent
    print("\n📚 AI KNOWLEDGE ASSISTANT")

    query = f"{event_type} event planning"

    knowledge_results = knowledge_agent(query)

    for result in knowledge_results:
        print("➡", result)

In [52]:
event_planner_system(
    event_type="wedding",
    city="Bangalore",
    budget=1000000
)

🎉 FESTIVA PLANNER AI

📋 EVENT PLAN
1. Book venue
2. Hire catering service
3. Arrange decorations
4. Send invitations
5. Book photographer

💰 BUDGET BREAKDOWN
venue: ₹350000
catering: ₹300000
decoration: ₹150000
entertainment: ₹100000
miscellaneous: ₹100000

📚 AI KNOWLEDGE ASSISTANT
➡ Start wedding planning at least 6 months before the event.
➡ Finalize wedding venue before invitations.


In [53]:
event = input("Enter event type: ")
city = input("Enter city: ")
budget = int(input("Enter budget: "))

event_planner_system(event, city, budget)

Enter event type: Wedding
Enter city: Bangalore
Enter budget: 1500000
🎉 FESTIVA PLANNER AI

📋 EVENT PLAN
1. Book venue
2. Hire catering service
3. Arrange decorations
4. Send invitations
5. Book photographer

💰 BUDGET BREAKDOWN
venue: ₹525000
catering: ₹450000
decoration: ₹225000
entertainment: ₹150000
miscellaneous: ₹150000

📚 AI KNOWLEDGE ASSISTANT
➡ Start wedding planning at least 6 months before the event.
➡ Finalize wedding venue before invitations.


In [96]:
!pip install -q fastapi nest-asyncio uvicorn



from fastapi import FastAPI
from pydantic import BaseModel

import nest_asyncio
import uvicorn




nest_asyncio.apply()




app = FastAPI()

print("✅ FastAPI App Created")




def planner_agent(event_type):

    plans = {
        "wedding": [
            "Book wedding venue",
            "Arrange catering",
            "Hire photographer",
            "Book decorators",
            "Send invitations"
        ],

        "birthday": [
            "Order birthday cake",
            "Buy decorations",
            "Arrange games",
            "Prepare food"
        ],

        "corporate": [
            "Book conference hall",
            "Arrange projector",
            "Invite speakers",
            "Prepare presentations"
        ]
    }

    return plans.get(
        event_type.lower(),
        ["No plan available"]
    )


def budget_agent(total_budget):

    return {
        "venue": int(total_budget * 0.35),
        "catering": int(total_budget * 0.30),
        "decoration": int(total_budget * 0.15),
        "entertainment": int(total_budget * 0.10),
        "miscellaneous": int(total_budget * 0.10)
    }


def knowledge_agent(event_type):

    knowledge = {
        "wedding":
            "Start planning at least 6 months before wedding.",

        "birthday":
            "Decorations and cake are most important.",

        "corporate":
            "Ensure proper audio and projector setup."
    }

    return knowledge.get(
        event_type.lower(),
        "No knowledge available"
    )




def event_planner_system(
    event_type,
    city,
    budget
):

    plan = planner_agent(event_type)

    budget_data = budget_agent(budget)

    knowledge = knowledge_agent(event_type)

    result = {

        "event_type": event_type,

        "city": city,

        "budget": budget,

        "event_plan": plan,

        "budget_breakdown": budget_data,

        "knowledge_assistant": knowledge
    }

    return result



class EventRequest(BaseModel):

    event_type: str
    city: str
    budget: int




@app.post("/plan_event")

def plan_event(request: EventRequest):

    result = event_planner_system(
        request.event_type,
        request.city,
        request.budget
    )

    return result




from threading import Thread

def run_api():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

api_thread = Thread(target=run_api)
api_thread.start()

print("✅ FastAPI running successfully")




output = event_planner_system(
    event_type="wedding",
    city="Bangalore",
    budget=1000000
)

print(output)

✅ FastAPI App Created
✅ FastAPI running successfully
{'event_type': 'wedding', 'city': 'Bangalore', 'budget': 1000000, 'event_plan': ['Book wedding venue', 'Arrange catering', 'Hire photographer', 'Book decorators', 'Send invitations'], 'budget_breakdown': {'venue': 350000, 'catering': 300000, 'decoration': 150000, 'entertainment': 100000, 'miscellaneous': 100000}, 'knowledge_assistant': 'Start planning at least 6 months before wedding.'}


INFO:     Started server process [47635]
INFO:     Waiting for application startup.
